In [0]:
basePath = "/Volumes/workspace/default/dataflowmed"
goldPath = f"{basePath}/gold"

def registerGoldView(tableName, viewName=None):
    viewName = viewName or tableName
    spark.read.format("delta").load(f"{goldPath}/{tableName}").createOrReplaceTempView(viewName)

registerGoldView("factOrderItems")
registerGoldView("factPayments")
registerGoldView("dimCustomers")
registerGoldView("dimProducts")
registerGoldView("dimSellers")

In [0]:
%sql
-- Q1: Monthly revenue trend with a running total (window function)
SELECT
orderMonth,
monthlyRevenue,
SUM(monthlyRevenue) OVER (
ORDER BY orderMonth
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) AS runningRevenue
FROM (
SELECT
date_format(order_purchase_timestamp, 'yyyy-MM') AS orderMonth,
SUM(total_item_value) AS monthlyRevenue
FROM factOrderItems
GROUP BY date_format(order_purchase_timestamp, 'yyyy-MM')
)
ORDER BY orderMonth

orderMonth,monthlyRevenue,runningRevenue
2016-09,354.75,354.75
2016-10,56808.84,57163.59
2016-12,19.62,57183.21
2017-01,137188.4899999999,194371.6999999999
2017-02,286280.62,480652.3199999999
2017-03,432048.59000000026,912700.9100000001
2017-04,412422.2400000005,1325123.1500000006
2017-05,586190.9500000024,1911314.100000003
2017-06,502963.0400000039,2414277.1400000066
2017-07,584971.620000004,2999248.7600000105


In [0]:
%sql
-- Q2: Top 10 products by revenue, joined to their category name
SELECT
p.product_id,
p.product_category,
SUM(f.total_item_value) AS totalRevenue,
COUNT(f.order_id) AS timesSold
FROM factOrderItems f
JOIN dimProducts p ON f.product_id = p.product_id
GROUP BY p.product_id, p.product_category
ORDER BY totalRevenue DESC
LIMIT 10

product_id,product_category,totalRevenue,timesSold
bb50f2e236e5eea0100680137654686c,health_beauty,67606.10000000003,195
d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,60976.02999999999,343
6cdd53843498f92890544667809f1595,health_beauty,59093.98999999993,156
99a4788cb24856965c36a24e339b6058,bed_bath_table,51071.59999999986,488
d6160fb7873f184099d9bc95e30376af,computers,50326.18000000001,35
3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,48212.22000000001,274
aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,44820.760000000235,527
5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,41725.81,63
25c38557cf793876c5abdd5931f922db,baby,40311.95,38
53b36df67ebb7c41585e8d54d6772e08,watches_gifts,39957.93000000011,323


In [0]:
%sql
-- Q3: Top 3 sellers per state (window function: RANK), via a CTE
WITH sellerRevenue AS (
SELECT
s.seller_id,
s.seller_state,
SUM(f.total_item_value) AS sellerRevenue
FROM factOrderItems f
JOIN dimSellers s ON f.seller_id = s.seller_id
GROUP BY s.seller_id, s.seller_state
),
rankedSellers AS (
SELECT
*,
RANK() OVER (PARTITION BY seller_state ORDER BY sellerRevenue DESC) AS rankInState
FROM sellerRevenue
)
SELECT *
FROM rankedSellers
WHERE rankInState <= 3
ORDER BY seller_state, rankInState

seller_id,seller_state,sellerRevenue,rankInState
4be2e7f96b4fd749d52dff41f80e39dd,AC,299.84000000000003,1
327b89b872c14d1c0be7235ef4871685,AM,1258.8,1
53243585a1d6dc2643021fd1853d8905,BA,235856.68000000058,1
c72de06d72748d1a0dfb2125be43ba63,BA,18733.18,2
75d34ebb1bd0bd7dde40dd507b8169c3,BA,16479.509999999995,3
bbf9ad41dca6603e614efcdad7aab8c4,CE,8622.259999999998,1
dbdd0ec73a4817971d962698f2fea022,CE,8174.8,2
8d79c8a04e42d722a75097ce5cbcf2ef,CE,3535.1599999999994,3
44073f8b7e41514de3b7815dd0237f4f,DF,21901.93,1
f3b80352b986ab4d1057a4b724be19d0,DF,12692.730000000003,2


In [0]:
%sql
-- Q4: Customer segmentation using CASE, based on number of orders placed
WITH customerOrderCounts AS (
SELECT
customer_id,
COUNT(DISTINCT order_id) AS orderCount
FROM factOrderItems
GROUP BY customer_id
),
customerSegments AS (
SELECT
customer_id,
orderCount,
CASE
WHEN orderCount = 1 THEN 'New'
WHEN orderCount BETWEEN 2 AND 4 THEN 'Regular'
ELSE 'Loyal'
END AS customerSegment
FROM customerOrderCounts
)
SELECT
customerSegment,
COUNT(customer_id) AS numCustomers
FROM customerSegments
GROUP BY customerSegment
ORDER BY numCustomers

customerSegment,numCustomers
New,98666


In [0]:
%sql
-- Q5: Payment type breakdown by value tier (CTE + CASE)
WITH paymentSummary AS (
SELECT
primary_payment_type,
total_payment_value,
CASE
WHEN total_payment_value < 100 THEN 'Low'
WHEN total_payment_value BETWEEN 100 AND 500 THEN 'Medium'
ELSE 'High'
END AS valueTier
FROM factPayments
)
SELECT
primary_payment_type,
valueTier,
COUNT(*) AS numOrders,
ROUND(SUM(total_payment_value), 2) AS totalValue
FROM paymentSummary
GROUP BY primary_payment_type, valueTier
ORDER BY totalValue DESC

primary_payment_type,valueTier,numOrders,totalValue
credit_card,Medium,37437,7231162.72
credit_card,High,3505,3235200.77
credit_card,Low,34033,2076163.66
boleto,Medium,8644,1611654.97
boleto,High,678,645074.69
boleto,Low,10462,612631.61
voucher,Medium,1177,206113.52
debit_card,Medium,631,117785.23
voucher,Low,1905,109563.71
voucher,High,69,63366.68


In [0]:
for tableName in ["factOrderItems", "factPayments", "dimCustomers", "dimProducts", "dimSellers"]:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {tableName}").collect()[0]["cnt"]
    print(f"{tableName}: {count} rows")

factOrderItems: 112650 rows
factPayments: 99440 rows
dimCustomers: 99444 rows
dimProducts: 32951 rows
dimSellers: 3095 rows
